# 🌿 PlantXAI — Full Training Run (Google Colab)

**Configuration:**
- 10 epochs × 5 split ratios (80:20, 75:25, 70:30, 65:35, 60:40) × 2 architectures = **100 training epochs**
- MobileNetV3-Small (~1.5M params) + EfficientNet-Lite0 (~4.7M params)
- Dual-model consensus (High/Medium/Low) computed automatically

**Before running:** Make sure you have selected a **GPU runtime** (Runtime → Change runtime type → T4 GPU)

## Step 0: Verify GPU & Print Hardware Info

In [ ]:
import torch
import platform
import subprocess

print("=" * 60)
print("  HARDWARE INFORMATION (for paper)")
print("=" * 60)

# GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"  GPU: {gpu_name} ({gpu_mem:.1f} GB)")
    print(f"  CUDA version: {torch.version.cuda}")
else:
    print("  ⚠️  NO GPU DETECTED! Go to Runtime → Change runtime type → T4 GPU")
    print("  Training will be extremely slow on CPU.")

# CPU
print(f"  CPU: {platform.processor() or 'Unknown'}")
print(f"  CPU threads (torch): {torch.get_num_threads()}")

# PyTorch
print(f"  PyTorch: {torch.__version__}")
print(f"  Python: {platform.python_version()}")
print(f"  OS: {platform.platform()}")

# nvidia-smi
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                            '--format=csv,noheader'], capture_output=True, text=True)
    print(f"  nvidia-smi: {result.stdout.strip()}")
except:
    pass

print("=" * 60)

## Step 1: Install Dependencies

In [ ]:
!pip install -q kagglehub timm scikit-learn seaborn pandas matplotlib

## Step 2: Upload train.py

Upload your patched `train.py` file when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload train.py here
print(f"\nUploaded: {list(uploaded.keys())}")

## Step 3: Run the Full Training

This will:
1. Download the 15-class PlantVillage dataset (20,638 images)
2. Remove 14 duplicates → 20,624 unique images
3. Train MobileNetV3-Small + EfficientNet-Lite0 across 5 splits × 10 epochs
4. Evaluate all metrics (Accuracy, Precision, Recall, F1, Macro-F1, AUROC)
5. Generate confusion matrices, per-class reports, saved predictions
6. Compute dual-model consensus (High/Medium/Low)
7. Save everything to `results/`

**Expected time: ~30-45 minutes on T4 GPU**

In [ ]:
!python train.py

## Step 4: Inspect Results

In [ ]:
import pandas as pd

# Main metrics table
print("\n" + "=" * 100)
print("  METRICS SUMMARY")
print("=" * 100)
df = pd.read_csv('results/metrics_summary.csv')
display(df.drop(columns=['note_latency'], errors='ignore').round(3))

# Consensus table
print("\n" + "=" * 100)
print("  DUAL-MODEL CONSENSUS")
print("=" * 100)
try:
    cons = pd.read_csv('results/consensus_summary.csv')
    display(cons.round(2))
except FileNotFoundError:
    print("  consensus_summary.csv not found.")

In [ ]:
# Display confusion matrices
import os
from IPython.display import display, Image as IPImage

cm_files = sorted([f for f in os.listdir('results') if f.startswith('cm_') and f.endswith('.png')])
for cm_file in cm_files:
    print(f"\n{cm_file}:")
    display(IPImage(filename=f'results/{cm_file}', width=700))

## Step 5: List All Output Files

In [ ]:
import os

print("\nFiles in results/:")
for f in sorted(os.listdir('results')):
    size = os.path.getsize(f'results/{f}')
    print(f"  {f:<50} {size:>10,} bytes")

print("\nWeight files (.pth):")
for f in sorted([f for f in os.listdir('.') if f.endswith('.pth')]):
    size = os.path.getsize(f)
    print(f"  {f:<50} {size:>10,} bytes")

print("\nConfig files:")
for f in ['class_names.json', 'ensemble_config.json']:
    if os.path.exists(f):
        print(f"  {f} ✓")

## Step 6: Zip & Download Everything

In [ ]:
import shutil
from google.colab import files

# Zip results/ folder
shutil.make_archive('plantxai_results', 'zip', '.', 'results')
print("Created: plantxai_results.zip")

# Also zip ALL weight files + configs for deployment
import zipfile
with zipfile.ZipFile('plantxai_weights.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in sorted([f for f in os.listdir('.') if f.endswith('.pth')]):
        zf.write(f)
    zf.write('class_names.json')
    zf.write('ensemble_config.json')
print("Created: plantxai_weights.zip")

# Download both
files.download('plantxai_results.zip')
files.download('plantxai_weights.zip')

## (Optional) Step 7: Upload & Run test.py for Consensus Evaluation

Upload `test.py` and call `evaluate_consensus()` to get the High/Medium/Low breakdown.

In [ ]:
# Upload test.py
from google.colab import files
uploaded = files.upload()  # Upload test.py here
print(f"Uploaded: {list(uploaded.keys())}")

In [ ]:
# Install grad-cam dependency (needed for test.py imports, even if we only use evaluate_consensus)
!pip install -q pytorch-grad-cam

# Run consensus evaluation
from test import evaluate_consensus
consensus_rows = evaluate_consensus()